In [2]:
import time
import pandas as pd
import chromedriver_autoinstaller

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, WebDriverException


# =========================
# 0. 경로 & 기본 설정
# =========================
INPUT_XLSX = "yeoti_hospital_details_all.xlsx"      # 병원 URL 목록 파일
OUTPUT_XLSX = "yeoti_hospital_details_with_name.xlsx"  # 병원명 추가해서 저장할 파일
OUTPUT_CSV  = "yeoti_hospital_details_with_name.csv"

WAIT_SEC = 10       # h1 기다리는 시간
SLEEP_BETWEEN = 1.0 # 요청 사이 딜레이


# =========================
# 1. 크롬 드라이버 설정
# =========================
def get_driver():
    chromedriver_autoinstaller.install()
    options = Options()
    # 디버깅 끝나면 주석 해제해서 headless로 돌려도 됨
    # options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--window-size=1920,1080")
    driver = webdriver.Chrome(options=options)
    return driver


# =========================
# 2. 병원명 추출 함수
# =========================
def get_hospital_name(driver, url):
    """
    여신티켓 병원 상세페이지에서 h1에 적힌 병원명만 가져옴.
    DOM 예시:
      <h1 class="font-semibold text-[22px] leading-[32px] break-words">세가지소원외과의원(압구정)</h1>
    """
    try:
        driver.get(url)
    except WebDriverException:
        return None

    try:
        # h1 중에서 font-semibold + break-words 포함된 요소를 타겟
        elem = WebDriverWait(driver, WAIT_SEC).until(
            EC.presence_of_element_located((
                By.XPATH,
                "//h1[contains(@class,'font-semibold') and contains(@class,'break-words')]"
            ))
        )
        name = elem.text.strip()
        if not name:
            return None
        return name
    except TimeoutException:
        return None


# =========================
# 3. 메인 루틴
# =========================
def main():
    # 1) 병원 URL 목록 로드
    df = pd.read_excel(INPUT_XLSX)
    if "hospital_url" not in df.columns:
        raise ValueError("엑셀에 'hospital_url' 컬럼이 없습니다.")

    urls = df["hospital_url"].astype(str)

    driver = get_driver()

    hospital_names = []
    errors = []

    for idx, url in enumerate(urls):
        url = url.strip()
        if not url or url.lower() == "nan":
            hospital_names.append(None)
            continue

        print(f"[{idx+1}/{len(urls)}] 병원명 수집 중: {url}")
        name = get_hospital_name(driver, url)
        hospital_names.append(name)

        if name is None:
            errors.append(url)
            print(f"  ⚠ 병원명 수집 실패")

        time.sleep(SLEEP_BETWEEN)

    driver.quit()

    # 2) 결과 병합 후 저장
    df["hospital_name"] = hospital_names
    df.to_excel(OUTPUT_XLSX, index=False)
    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

    print("\n✅ 병원명 수집 완료")
    print(f"저장 파일: {OUTPUT_XLSX}, {OUTPUT_CSV}")
    print(f"수집 실패 URL 개수: {len(errors)}")
    if errors:
        print("실패 URL 예시 5개:")
        for u in errors[:5]:
            print(" -", u)


if __name__ == "__main__":
    main()


[1/367] 병원명 수집 중: https://www.yeoshin.co.kr/hospitals/119?from=events&tabmenu=INFO
[2/367] 병원명 수집 중: https://www.yeoshin.co.kr/hospitals/103860?from=events&tabmenu=INFO
[3/367] 병원명 수집 중: https://www.yeoshin.co.kr/hospitals/5408?from=events&tabmenu=INFO
[4/367] 병원명 수집 중: https://www.yeoshin.co.kr/hospitals/4157?from=events&tabmenu=INFO
[5/367] 병원명 수집 중: https://www.yeoshin.co.kr/hospitals/103999?from=events&tabmenu=INFO
[6/367] 병원명 수집 중: https://www.yeoshin.co.kr/hospitals/5241?from=events&tabmenu=INFO
[7/367] 병원명 수집 중: https://www.yeoshin.co.kr/hospitals/4517?from=events&tabmenu=INFO
[8/367] 병원명 수집 중: https://www.yeoshin.co.kr/hospitals/5310?from=events&tabmenu=INFO
[9/367] 병원명 수집 중: https://www.yeoshin.co.kr/hospitals/4696?from=events&tabmenu=INFO
[10/367] 병원명 수집 중: https://www.yeoshin.co.kr/hospitals/85450?from=events&tabmenu=INFO
[11/367] 병원명 수집 중: https://www.yeoshin.co.kr/hospitals/5346?from=events&tabmenu=INFO
[12/367] 병원명 수집 중: https://www.yeoshin.co.kr/hospitals/4917?from=event